# Vector Spaces u AI: Praktican Colab projekat

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/boba1987/vector-spaces/blob/master/vector_spaces_ai_colab.ipynb)

Ovaj notebook pokazuje kako se teorija iz **Gilbert Strang, Chapter 2: Vector Spaces** primenjuje u AI zadatku semantic retrieval-a.

## Sta demonstriramo
- podprostore i projekcije (`2.1 Vector Spaces and Subspaces`)
- resavanje kroz decomposition ideju (`2.2 Solving Ax=0 and Ax=b`)
- bazu i dimenziju (`2.3 Linear Independence, Basis, and Dimension`)
- ulogu fundamentalnih podprostora (`2.4 The Four Fundamental Subspaces`)

## Reference iz dokumenta
- PDF: `Gilbert_Strang_Linear_Algebra_and_Its_Applications.pdf`
- Mapirani deo: **Chapter 2 Vector Spaces**, PDF strane oko **87-130** (book pagination ~77-120)
- Sekcije: `2.1`, `2.2`, `2.3`, `2.4`

Cilj: pokazati kako projekcija na podprostor moze poboljsati signal/noise u retrieval-u i kako `nullspace` komponenta moze meriti novinu query-ja.

In [ ]:
# Colab setup: osnovne biblioteke za numeriku, ML i crtanje.
# Ako pokreces lokalno, ova celija je i dalje bezbedna.

!pip -q install numpy pandas matplotlib scikit-learn

In [ ]:
# Importi i reproduktivnost.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

SEED = 42
np.random.seed(SEED)

print('Setup OK. Seed =', SEED)

## Logging tokom izvrsavanja

U sledecim celijama uvodimo helper funkcije za citljiv log:
- koji korak je izvrsen
- koje su dimenzije matrica
- kratki preview kako matrice izgledaju

In [ ]:
from IPython.display import display


def log_step(step_name, details=''):
    print(f'\n[LOG] {step_name}')
    if details:
        print(f'      {details}')


def log_matrix(name, M, preview_rows=3, preview_cols=6):
    # Jedinstven log format za sve matrice.
    print(f'[MATRIX] {name}: shape={M.shape}, dtype={M.dtype}')
    r = min(preview_rows, M.shape[0])
    c = min(preview_cols, M.shape[1]) if len(M.shape) > 1 else 1
    if len(M.shape) == 1:
        print(f'[MATRIX] {name} preview:', M[:r])
    else:
        preview_df = pd.DataFrame(M[:r, :c])
        display(preview_df)


log_step('Logger inicijalizovan', 'Notebook ce ispisivati sta je napravljeno i kako izgleda.')

## 1) Mini dataset za praktican AI primer

Koristicemo malu kolekciju dokumenata sa vise tema. Time simuliramo realan retrieval problem gde nisu svi tekstovi iz iste oblasti.

In [ ]:
# Dokumenti i label-e (tema dokumenta).
# Ove label-e koristimo za evaluaciju Recall@k.

documents = [
    'Transformers use self-attention to model long-range dependencies in text.',
    'Gradient descent minimizes a loss function by iterative parameter updates.',
    'A vector space is closed under addition and scalar multiplication.',
    'The nullspace contains all vectors x such that Ax equals zero.',
    'PCA projects data onto principal components with maximum variance.',
    'Convolutional neural networks are effective for image classification tasks.',
    'Soccer teams optimize passing networks to create scoring opportunities.',
    'A healthy diet includes vegetables, proteins, and balanced micronutrients.',
    'Linear independence means no vector is a combination of others.',
    'Basis vectors span a subspace and define its dimension.',
    'Stock market volatility can increase during macroeconomic uncertainty.',
    'Strength training improves muscle mass and insulin sensitivity.'
]

labels = [
    'ai', 'ai', 'linear_algebra', 'linear_algebra', 'ai', 'ai',
    'sports', 'health', 'linear_algebra', 'linear_algebra', 'finance', 'health'
]

df_docs = pd.DataFrame({'doc_id': range(len(documents)), 'text': documents, 'label': labels})

log_step('Dataset pripremljen', f'Broj dokumenata: {len(documents)} | Broj labela: {len(set(labels))}')
display(df_docs.head())

## 2) Embedding matrica X i veza sa teorijom (2.1, 2.3)

U jeziku linearne algebre, svaku recenicu mapiramo u vektor. Slaganjem svih vektora dobijamo matricu `X`.

- Kolone matrice predstavljaju feature prostor.
- Rang i broj nezavisnih pravaca daju intuitivnu sliku kapaciteta reprezentacije.
- U `2.3` ovo odgovara idejama baza i dimenzija.

In [ ]:
# Primarni embedding backend: TF-IDF (lagan, pouzdan za Colab).
# Opcioni backend: sentence-transformers (mozes ukljuciti ako zelis semanticki jace embeddinge).

USE_SENTENCE_TRANSFORMERS = False

if USE_SENTENCE_TRANSFORMERS:
    !pip -q install sentence-transformers
    from sentence_transformers import SentenceTransformer
    model = SentenceTransformer('all-MiniLM-L6-v2')
    X = model.encode(documents, convert_to_numpy=True)
    feature_names = [f'emb_{i}' for i in range(X.shape[1])]
    log_step('Embedding backend', 'Koristim sentence-transformers embeddinge.')
else:
    vectorizer = TfidfVectorizer(stop_words='english')
    X_sparse = vectorizer.fit_transform(documents)
    X = X_sparse.toarray()
    feature_names = vectorizer.get_feature_names_out().tolist()
    log_step('Embedding backend', 'Koristim TF-IDF embeddinge.')

log_step('Embedding matrica kreirana', f'Oblik X={X.shape} | Rang(X)~{np.linalg.matrix_rank(X)}')
log_matrix('X', X)

## 3) Podprostor, baza i dimenzija preko SVD (2.1, 2.3)

Koristimo SVD da izdvojimo dominantne pravce varijacije.

Ako uzmemo prvih `k` singularnih vektora, oni formiraju bazu podprostora dimenzije `k`.
To je prakticna verzija ideja iz sekcija `2.1` i `2.3`.

In [ ]:
# SVD dekompozicija: X = U S V^T
# Desni singularni vektori (kolone V) daju ortonormisanu bazu feature prostora.

U, S, Vt = np.linalg.svd(X, full_matrices=False)

# Biramo dimenziju podprostora k (mozes menjati i posmatrati trade-off).
k = min(5, X.shape[1])
V_k = Vt[:k].T  # baza podprostora u feature prostoru, dimenzije (d, k)

# Objasnjena energija kao prakticna intuicija koliko informacije cuvamo.
energy_ratio = np.sum(S[:k] ** 2) / np.sum(S ** 2)

log_step('SVD podprostor izracunat', f'k={k} | captured_energy={energy_ratio:.4f}')
log_matrix('V_k (baza podprostora)', V_k)
log_matrix('Singular values (kao vektor)', S)

## 4) Projekcija i nullspace komponenta (2.2, 2.4)

Za vektor `x`:
- projekcija na podprostor: `x_proj = x V_k V_k^T`
- residual (komponenta van podprostora): `x_null_like = x - x_proj`

Intuicija iz `2.2` i `2.4`:
- deo signala koji prostor "objasnjava" ostaje u projekciji
- deo koji ne objasnjava ostaje u residual-u (ovde ga koristimo kao signal novine)

Napomena: u strogoj mat. notaciji za opsti slucaj govorimo o ortogonalnom komplementu, a ne uvek o exact `N(A)`. U AI praksi ovaj residual cesto sluzi kao nullspace-like signal.

In [ ]:
# Funkcije za projekciju i merenje residual-a.

def project_to_subspace(X_in, basis):
    # X_in: (n, d), basis: (d, k)
    return X_in @ basis @ basis.T


def residual_component(X_in, X_proj):
    return X_in - X_proj


X_proj = project_to_subspace(X, V_k)
X_res = residual_component(X, X_proj)

# Prosecna jacina residual komponente po dokumentu.
res_norms = np.linalg.norm(X_res, axis=1)

log_step('Projekcija i residual izracunati', f'Mean residual norm={float(np.mean(res_norms)):.4f}')
log_matrix('X_proj', X_proj)
log_matrix('X_res', X_res)
display(pd.DataFrame({'doc_id': df_docs.doc_id, 'label': df_docs.label, 'residual_norm': res_norms}).head())

## 5) Retrieval: baseline vs subspace-aware

- **Baseline**: cosine slicnost u originalnom embedding prostoru.
- **Subspace-aware**: cosine slicnost nakon projekcije na podprostor.

Time testiramo da li fokusiranje na dominantni podprostor popravlja pretragu.

In [ ]:
# Definisemo upite sa expected klasom radi evaluacije.
queries = [
    ('How do attention mechanisms work in transformers?', 'ai'),
    ('What defines a basis and vector space dimension?', 'linear_algebra'),
    ('Tips for balanced nutrition and healthy meals', 'health'),
    ('How to track macroeconomic market volatility', 'finance')
]


def encode_queries(query_texts):
    # Koristimo isti backend kao za dokumente.
    if USE_SENTENCE_TRANSFORMERS:
        return model.encode(query_texts, convert_to_numpy=True)
    return vectorizer.transform(query_texts).toarray()


def topk_indices(sim_row, topk=3):
    return np.argsort(-sim_row)[:topk]


Q = encode_queries([q for q, _ in queries])
Q_proj = project_to_subspace(Q, V_k)
Q_res = residual_component(Q, Q_proj)

sim_baseline = cosine_similarity(Q, X)
sim_subspace = cosine_similarity(Q_proj, X_proj)

# Novelty signal query-ja: veci residual moze znaciti da je query van dominantnog podprostora.
q_novelty = np.linalg.norm(Q_res, axis=1)

log_step('Query retrieval izracunat', f'Broj query-ja: {len(queries)}')
log_matrix('Q', Q)
log_matrix('Q_proj', Q_proj)

for i, (q_text, expected) in enumerate(queries):
    b_idx = topk_indices(sim_baseline[i], topk=3)
    s_idx = topk_indices(sim_subspace[i], topk=3)

    print('\n[QUERY LOG]', q_text)
    print('Expected label:', expected)
    print(f'Novelty score (residual norm): {q_novelty[i]:.4f}')
    print('Baseline top-3 labels :', df_docs.iloc[b_idx].label.tolist())
    print('Subspace top-3 labels :', df_docs.iloc[s_idx].label.tolist())

## 6) Evaluacija: Recall@k i trend po dimenziji podprostora

Ovde kvantifikujemo koliko cesto sistem vraca dokument iste klase kao query.
To je jednostavna, ali korisna retrieval metrika za demonstraciju.

In [ ]:
def recall_at_k(sim_matrix, true_labels, doc_labels, k=3):
    hits = 0
    for i in range(sim_matrix.shape[0]):
        idx = topk_indices(sim_matrix[i], topk=k)
        retrieved_labels = [doc_labels[j] for j in idx]
        if true_labels[i] in retrieved_labels:
            hits += 1
    return hits / sim_matrix.shape[0]


true_labels = [lbl for _, lbl in queries]
doc_labels = df_docs.label.tolist()

baseline_r3 = recall_at_k(sim_baseline, true_labels, doc_labels, k=3)
subspace_r3 = recall_at_k(sim_subspace, true_labels, doc_labels, k=3)

log_step('Evaluacija zavrsena', f'Recall@3 baseline={baseline_r3:.4f} | subspace={subspace_r3:.4f}')

In [ ]:
# Analiza uticaja dimenzije k na Recall@3 i captured energy.
# Ovo je direktan practical trade-off izmedju kompresije i gubitka informacije.

k_values = list(range(1, min(8, X.shape[1]) + 1))
recalls = []
energies = []

for k_try in k_values:
    V_try = Vt[:k_try].T
    Xp_try = project_to_subspace(X, V_try)
    Qp_try = project_to_subspace(Q, V_try)

    sim_try = cosine_similarity(Qp_try, Xp_try)
    recalls.append(recall_at_k(sim_try, true_labels, doc_labels, k=3))

    e_try = np.sum(S[:k_try] ** 2) / np.sum(S ** 2)
    energies.append(e_try)

fig, ax1 = plt.subplots(figsize=(8, 4))
ax1.plot(k_values, recalls, marker='o', label='Recall@3')
ax1.set_xlabel('Subspace dimension k')
ax1.set_ylabel('Recall@3')
ax1.set_ylim(0, 1.05)

ax2 = ax1.twinx()
ax2.plot(k_values, energies, marker='s', linestyle='--', label='Captured energy')
ax2.set_ylabel('Captured spectral energy')
ax2.set_ylim(0, 1.05)

ax1.set_title('Trade-off: subspace dimension vs retrieval quality')
plt.show()

## 8) Finalni log artefakata

Ova celija daje sazet pregled sta je napravljeno tokom run-a i kako izgleda izlazni artefakt.

In [ ]:
summary_df = pd.DataFrame([
    {'artifact': 'documents', 'shape_or_value': len(documents), 'notes': 'Broj ulaznih tekstova'},
    {'artifact': 'X', 'shape_or_value': str(X.shape), 'notes': 'Embedding matrica'},
    {'artifact': 'V_k', 'shape_or_value': str(V_k.shape), 'notes': 'Baza podprostora'},
    {'artifact': 'X_proj', 'shape_or_value': str(X_proj.shape), 'notes': 'Projekcija dokumenata'},
    {'artifact': 'Q', 'shape_or_value': str(Q.shape), 'notes': 'Matrica query-ja'},
    {'artifact': 'Q_proj', 'shape_or_value': str(Q_proj.shape), 'notes': 'Projekcija query-ja'},
    {'artifact': 'baseline Recall@3', 'shape_or_value': f'{baseline_r3:.4f}', 'notes': 'Baseline retrieval'},
    {'artifact': 'subspace Recall@3', 'shape_or_value': f'{subspace_r3:.4f}', 'notes': 'Subspace-aware retrieval'}
])

log_step('Finalni summary log spreman', 'Ispod je tabela svih glavnih artefakata.')
display(summary_df)

## 7) Zakljucak i mapiranje na teoriju

- `2.1`: Dokument embeddingi prirodno formiraju vektorski prostor; biramo podprostor od interesa.
- `2.2`: Razdvajanje na "objasnjeni" deo (projekcija) i residual (nullspace-like signal) je prakticna analogija `Ax=b` + homogeni deo.
- `2.3`: `k` i baza (`V_k`) eksplicitno kontrolisu dimenziju reprezentacije.
- `2.4`: Uvid u rang, prostor kolona i ortogonalne komponente pomaze da razumemo kada model gubi/filtrira informaciju.

### Practical takeaway
Subspace-aware retrieval je koristan kada zelis robusniji signal i kontrolisanu kompresiju, dok residual norma moze sluziti kao indikator da je query van dominantnog domena podataka.